In [ ]:
from pathlib import Path
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

ROOT = Path().resolve().parent
sys.path.append(str(ROOT))

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
def get_spark():
    return SparkSession.builder \
        .appName('TCC - Testes') \
        .getOrCreate()

spark = get_spark()

In [ ]:
base_path = "../data/criminalidade/curated/veiculos_setor_parquet"

df = (
    spark.read
    .option("basePath", base_path)
    .option("mergeSchema", "true")
    .parquet(f"{base_path}/ano_trimestre=*")
)

df_ocorrencias = df.withColumn(
    "ano",
    F.regexp_extract(
        F.col("ano_trimestre"),
        r"(\d{4})",
        1
    ).cast("int")
)

df_ocorrencias = (
    df_ocorrencias
    .filter(
        F.col("ano").cast("int").between(2020, 2025)
    )
)

df_ocorrencias = df_ocorrencias.filter(df_ocorrencias.SJ_CD_SETOR.isNotNull())
# df_ocorrencias.printSchema()
df_ocorrencias.show(5, False)

In [ ]:
df_ocorrencias.createorReplaceTempView('teste')

spark.sql("""select ano, count(*) qt from teste groupm by 1 order by 1 desc""").show()